# Import des bibliothèques nécessaires

In [ ]:
import pandas as pd #pour la manipulation de données
import numpy as np
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from collections import Counter



# Chargement des données

In [2]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "01_paquets_128.csv", encoding="utf-8",)
df.head()

,nom_fichier,id_paquet,phrases_paquet,nb_tokens_camembert
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...,122
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Debout devant l’armoire, en face des fenêtres,...",93
2,1893_20_Le_docteur_Pascal._clean.txt,2,Il y avait plus de trente ans que le docteur y...,107
3,1893_20_Le_docteur_Pascal._clean.txt,3,"Lui-même, dans cette clarté d’aube, apparaissa...",112
4,1893_20_Le_docteur_Pascal._clean.txt,4,Jamais Ramond ne déchiffrerait ma satanée écri...,114


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61435 entries, 0 to 61434
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   nom_fichier          61435 non-null  str  
 1   id_paquet            61435 non-null  int64
 2   phrases_paquet       61435 non-null  str  
 3   nb_tokens_camembert  61435 non-null  int64
dtypes: int64(2), str(2)
memory usage: 1.9 MB


# Préparation du dataframe 

In [4]:
df = df.rename(columns={
    "nom_fichier": "fichier",
    "id_paquet": "paquet_id",
    "phrases_paquet": "texte"
})

In [5]:
df["annee"] = df["fichier"].str.extract(r"^(\d{4})").astype(int)

In [6]:
ordre_romans= (
    df[["fichier", "annee"]]
    .drop_duplicates()
    .sort_values(["annee", "fichier"])
    .reset_index(drop=True)
)

ordre_romans["ordre_romans"] = range(1, len(ordre_romans) + 1)

df = df.merge(ordre_romans[["fichier", "ordre_romans"]], on="fichier", how="left")

In [7]:
df["roman"] = (
    df["fichier"]
    .str.replace(r"^\d{4}_\d+_", "", regex=True)
    .str.replace(r"_clean\.txt$", "", regex=True)
    .str.replace("_", " ")
)

In [8]:
df = df[[
    "roman",
    "annee",
    "ordre_romans",
    "paquet_id",
    "texte",
    "nb_tokens_camembert"
]]

In [9]:
df = df.sort_values(["ordre_romans", "paquet_id"]).reset_index(drop=True)

In [10]:
df["paquet_id"] = df.groupby("roman").cumcount() + 1

In [11]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_tokens_camembert
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",105
1,1865 La confession de Claude.,1865,1,2,"Mon grenier, tout au haut d’un escalier humide...",124
2,1865 La confession de Claude.,1865,1,3,"Le soir, quand le vent ébranle la porte et que...",114
3,1865 La confession de Claude.,1865,1,4,Le foyer demande de grands feux joyeux; les va...,113
4,1865 La confession de Claude.,1865,1,5,Alors elle me paraît plus déserte et plus misé...,100


# Lemmatisation des données

In [ ]:
POS_AUTORISEES = {"NOUN", "ADJ", "VERB"}

# Le parser et le NER ne sont pas utilisés.
# "exclude" évite aussi de charger leurs données en mémoire.
nlp = spacy.load("fr_core_news_lg",exclude=["parser", "ner"])

print("Composants actifs :", nlp.pipe_names)

if df["texte"].isna().any():
    raise ValueError(f"{df['texte'].isna().sum()} textes sont manquants.")
textes_bruts = df["texte"].tolist()
textes_nettoyes = []

for doc in nlp.pipe(textes_bruts,batch_size=256,n_process=2):
    lemmes = []

    for token in doc:
        lemme = token.lemma_.casefold().strip()

        if (
            token.pos_ in POS_AUTORISEES
            and not token.is_punct
            and not token.like_num
            and not token.is_space
            and len(lemme) >= 2
        ):
            lemmes.append(lemme)

    textes_nettoyes.append(" ".join(lemmes))

df["phrases_lemm"] = textes_nettoyes

Composants actifs : ['tok2vec', 'morphologizer', 'attribute_ruler', 'lemmatizer']


In [13]:
nb_lemmes = df["phrases_lemm"].str.split().str.len()

print(nb_lemmes.describe())
print("Segments vides :", (nb_lemmes == 0).sum())

#assert len(textes_nettoyes) == len(df)
#assert df["phrases_lemm"].str.strip().ne("").all()

count    61435.000000
mean        29.273118
std          6.686890
min          0.000000
25%         25.000000
50%         30.000000
75%         34.000000
max         50.000000
Name: phrases_lemm, dtype: float64
Segments vides : 3


In [14]:
masque_vides = (
    df["phrases_lemm"]
    .fillna("")
    .str.strip()
    .eq("")
)

colonnes = [
    colonne
    for colonne in [
        "roman",
        "paquet_id",
        "texte",
        "phrases_lemm"
    ]
    if colonne in df.columns
]

display(
    df.loc[masque_vides, colonnes]
)

,roman,paquet_id,texte,phrases_lemm
23897,Au Bonheur des dames.,972,Ah!,
42045,Le docteur Pascal.,542,Ah!,
46892,Rome.,1104,Oh!,


In [ ]:
for index in df.index[masque_vides]:
    texte = df.at[index, "texte"]
    doc = nlp(texte)

    repartition_pos = Counter(
        token.pos_
        for token in doc
        if not token.is_punct
        and not token.is_space
        and not token.like_num
    )

    print(f"\nIndex : {index}")
    print("Texte :", texte)
    print("Catégories grammaticales :", repartition_pos)


Index : 23897
Texte : Ah!
Catégories grammaticales : Counter({'ADV': 1})

Index : 42045
Texte : Ah!
Catégories grammaticales : Counter({'ADV': 1})

Index : 46892
Texte : Oh!
Catégories grammaticales : Counter({'ADV': 1})


In [16]:
segments_exclus = df.loc[masque_vides].copy()

df = (
    df.loc[~masque_vides]
    .copy()
    .reset_index(drop=True)
)

assert df["phrases_lemm"].str.strip().ne("").all()

print("Segments conservés :", len(df))
print("Segments exclus :", len(segments_exclus))

Segments conservés : 61432
Segments exclus : 3


In [21]:
df.head(3)

,roman,annee,ordre_romans,paquet_id,texte,nb_tokens_camembert,phrases_lemm
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",105,voici hiver air matin devenir frais mettre man...
1,1865 La confession de Claude.,1865,1,2,"Mon grenier, tout au haut d’un escalier humide...",124,grenier haut escalier humide grand irrégulier ...
2,1865 La confession de Claude.,1865,1,3,"Le soir, quand le vent ébranle la porte et que...",114,soir vent ébranler porte mur vaciller flamme l...


In [20]:
df["paquet_id"].describe()

count    61432.000000
mean      1124.357371
std        763.391227
min          1.000000
25%        496.000000
50%       1023.500000
75%       1611.000000
max       3511.000000
Name: paquet_id, dtype: float64

In [22]:
chemin_sortie = Path("..") /"data" /"2_processed" /"02_corpus_zola_lematise_128.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")